# Gold Layer & Feature Engineering

## Objective

Create business-friendly features from the cleaned Silver dataset to improve predictive modeling and business insights.

Input Table:
workspace.default.customer_churn_silver

Output Table:
workspace.default.customer_churn_gold

In [ ]:
# Local Jupyter / GitHub compatibility (no-op on Databricks)
import os
import sys
from pathlib import Path

if "DATABRICKS_RUNTIME_VERSION" not in os.environ:
    _root = next(
        (
            p
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src" / "utils" / "databricks_compat.py").exists()
        ),
        None,
    )
    if _root and str(_root / "src") not in sys.path:
        sys.path.insert(0, str(_root / "src"))
    from utils.databricks_compat import inject_globals

    inject_globals(globals())


In [ ]:

from pyspark.sql import functions as F

In [ ]:

silver_df = spark.table(
    "workspace.default.customer_churn_silver"
)

display(silver_df.limit(5))

In [ ]:

print("=" * 60)
print("SILVER LAYER LOADED")
print("=" * 60)

print(f"Rows    : {silver_df.count()}")
print(f"Columns : {len(silver_df.columns)}")

In [ ]:

gold_df = silver_df

In [ ]:

gold_df = gold_df.fillna({
    "TotalCharges": 0.0
})

In [ ]:

print(
    gold_df.filter(
        F.col("TotalCharges").isNull()
    ).count()
)


### Features

In [ ]:

gold_df = gold_df.withColumn(
    "HasFamily",
    F.when(
        (F.col("Partner") == "Yes") |
        (F.col("Dependents") == "Yes"),
        "Yes"
    ).otherwise("No")
)

In [ ]:

display(
    gold_df.groupBy("HasFamily").count()
)

In [ ]:

gold_df = gold_df.withColumn(
    "TenureGroup",
    F.when(F.col("tenure") < 12, "New Customer")
     .when(F.col("tenure") < 36, "Regular Customer")
     .when(F.col("tenure") < 60, "Loyal Customer")
     .otherwise("Very Loyal Customer")
)

In [ ]:

display(
    gold_df.groupBy("TenureGroup").count()
)

In [ ]:

gold_df = gold_df.withColumn(
    "MonthlyChargeLevel",
    F.when(F.col("MonthlyCharges") < 35, "Low")
     .when(F.col("MonthlyCharges") < 70, "Medium")
     .otherwise("High")
)

In [ ]:

display(
    gold_df.groupBy("MonthlyChargeLevel").count()
)

In [ ]:

gold_df = gold_df.withColumn(
    "IsLongTermContract",
    F.when(
        F.col("Contract") == "Month-to-month",
        "No"
    ).otherwise("Yes")
)

In [ ]:

display(
    gold_df.groupBy("IsLongTermContract").count()
)

In [ ]:

gold_df = gold_df.withColumn(
    "HasInternet",
    F.when(
        F.col("InternetService") == "No",
        "No"
    ).otherwise("Yes")
)

In [ ]:

display(
    gold_df.groupBy("HasInternet").count()
)

In [ ]:

gold_df = gold_df.withColumn(
    "TotalServicesSubscribed",
    (
        F.when(F.col("PhoneService") == "Yes", 1).otherwise(0) +
        F.when(F.col("OnlineSecurity") == "Yes", 1).otherwise(0) +
        F.when(F.col("OnlineBackup") == "Yes", 1).otherwise(0) +
        F.when(F.col("DeviceProtection") == "Yes", 1).otherwise(0) +
        F.when(F.col("TechSupport") == "Yes", 1).otherwise(0) +
        F.when(F.col("StreamingTV") == "Yes", 1).otherwise(0) +
        F.when(F.col("StreamingMovies") == "Yes", 1).otherwise(0)
    )
)

In [ ]:

display(
    gold_df.select("TotalServicesSubscribed")
)

In [ ]:

gold_table = "workspace.default.customer_churn_gold"

(
    gold_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(gold_table)
)

print("=" * 60)
print(" GOLD LAYER CREATED SUCCESSFULLY")
print("=" * 60)
print(f"Table : {gold_table}")

In [ ]:

gold_df = spark.table(gold_table)

print("=" * 60)
print(" GOLD TABLE LOADED")
print("=" * 60)
print(f"Rows    : {gold_df.count()}")
print(f"Columns : {len(gold_df.columns)}")

In [ ]:

gold_df.printSchema()

In [ ]:

print("=" * 60)
print(" GOLD LAYER VALIDATION")
print("=" * 60)

print(f"Rows                : {gold_df.count()}")
print(f"Columns             : {len(gold_df.columns)}")
print(f"Duplicate Rows      : {gold_df.count() - gold_df.dropDuplicates().count()}")
print(f"Null TotalCharges   : {gold_df.filter('TotalCharges IS NULL').count()}")